# Capítulo 1 · La interfaz de una herramienta

Un agente para la Clínica Veterinaria Naranco. El agente y el modelo son siempre los mismos; lo que cambia son las herramientas que recibe. Se parte de la API interna de la clínica tal cual y se mejora en tres pasos: nombres y descripciones claros, resultados pensados para el modelo y herramientas con la forma de las tareas. Cada juego se mide con doce tareas de la clínica, comprobadas por la respuesta y por el estado de la agenda y de los avisos.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo y los datos de la clínica

El modelo ocupa unos 8 GB; la primera vez tarda unos minutos en descargarse. Los datos son nueve archivos JSON inventados: dueños, mascotas, historial, vacunas, citas, farmacia, tarifas, facturas y veterinarios.

In [ ]:
!pip install -q transformers==5.19.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 1: la interfaz de una herramienta. Las mismas tareas de la clínica, con el
mismo agente, y cuatro juegos de herramientas que van mejorando paso a paso."""
import copy
import json
import re
import time
import urllib.request
from datetime import date
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
MODELO = None                               # None: elegir según el entorno
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/"
HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def descargar(archivo):
    """Un archivo JSON de la carpeta datos; en Colab, se descarga."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return json.loads(ruta.read_text(encoding="utf-8"))

In [ ]:
tok, modelo = cargar(MODELO)

In [ ]:
def reiniciar():
    """Deja la clínica como al principio: la agenda original y ningún aviso."""
    global CITAS, AVISOS
    CITAS, AVISOS = copy.deepcopy(CITAS0), []


def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar nombres escritos de cualquier forma."""
    return str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu")).strip()


def cargar_datos():
    global MASCOTAS, DUENOS, HISTORIAL, VACUNAS, CITAS0, FARMACIA, FACTURAS
    global VETERINARIOS, HORAS
    MASCOTAS, DUENOS = descargar("mascotas.json"), descargar("duenos.json")
    HISTORIAL, VACUNAS = descargar("historial.json"), descargar("vacunas.json")
    CITAS0, FARMACIA = descargar("citas.json"), descargar("farmacia.json")
    FACTURAS, VETERINARIOS = descargar("facturas.json"), descargar("veterinarios.json")
    HORAS = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reiniciar()

In [ ]:
cargar_datos()
print(len(DUENOS), "clientes,", len(MASCOTAS), "mascotas y", len(CITAS0), "citas")
print(VETERINARIOS)

## 2. La API interna de la clínica

Así la escribieron los programadores de la aplicación web. Las tres funciones que actúan usan las de la clínica (`reservar_cita`, `cancelar_cita` y `avisar`), que son las mismas para los cuatro juegos de herramientas.

In [ ]:
def con_auditoria(registro, n):
    """Los campos de control que la base de datos añade a cada registro."""
    return {**registro, "created_at": f"2026-0{1 + n % 9}-1{n % 10}T09:{10 + n % 50}:00Z",
            "updated_by": f"usr_{100 + n % 7}", "sync": True, "rev": n % 5 + 1}


def get_pac(pid):
    """Get patient."""
    if pid not in MASCOTAS:
        return {"status": 404}
    return {"status": 200, "id": pid, **con_auditoria(MASCOTAS[pid], 1),
            "historial": [con_auditoria(v, i) for i, v in enumerate(HISTORIAL[pid])],
            "vacunas": [con_auditoria(v, i) for i, v in enumerate(VACUNAS[pid])],
            "dueno": {"id": MASCOTAS[pid]["dueno"],
                      **con_auditoria(DUENOS[MASCOTAS[pid]["dueno"]], 2)}}


def get_cli(cid):
    """Get client."""
    if cid not in DUENOS:
        return {"status": 404}
    mascotas = {m: d for m, d in MASCOTAS.items() if d["dueno"] == cid}
    return {"status": 200, "id": cid, **con_auditoria(DUENOS[cid], 3),
            "mascotas": {m: con_auditoria(d, 4) for m, d in mascotas.items()},
            "facturas": [con_auditoria(f, i) for i, f in enumerate(FACTURAS)
                         if f["dueno"] == cid]}


def qry(tabla, campo, valor):
    """Query table."""
    filas = {"mascotas": MASCOTAS, "duenos": DUENOS, "medicamentos": FARMACIA}.get(tabla)
    if filas is None:
        return {"status": 400}
    return {"status": 200, "rows": [{"id": i, **f} for i, f in filas.items()
                                    if sin_tildes(valor) in sin_tildes(f.get(campo, ""))]}


def agenda_dia(f):
    """Agenda."""
    return {"status": 200, "citas": [con_auditoria(c, i) for i, c in enumerate(CITAS)
                                     if c["fecha"] == f]}


def add_cita(pid, vid, f, h, mot):
    """New appointment."""
    return reservar_cita(pid, vid, f, h, mot)


def del_cita(cid):
    """Delete appointment."""
    return cancelar_cita(cid)


def send_msg(cid, txt):
    """Send message."""
    return avisar(cid, txt)


JUEGO_API = {"get_pac": get_pac, "get_cli": get_cli, "qry": qry,
             "agenda_dia": agenda_dia, "add_cita": add_cita, "del_cita": del_cita,
             "send_msg": send_msg}

In [ ]:
def reservar_cita(mascota, veterinario, fecha, hora, motivo):
    """Reserva un hueco libre de la agenda; devuelve la cita o el problema."""
    if mascota not in MASCOTAS:
        return {"error": f"No existe la mascota {mascota}."}
    if veterinario not in VETERINARIOS:
        return {"error": f"No existe el veterinario {veterinario}."}
    if any(c["fecha"] == fecha and c["hora"] == hora and c["veterinario"] == veterinario
           and c["estado"] != "cancelada" for c in CITAS):
        return {"error": f"{VETERINARIOS[veterinario]} ya tiene una cita el {fecha} "
                         f"a las {hora}."}
    cita = {"id": f"CT-{len(CITAS0) + len(CITAS) + 1:04d}", "fecha": fecha, "hora": hora,
            "veterinario": veterinario, "mascota": mascota, "motivo": motivo,
            "estado": "pendiente"}
    CITAS.append(cita)
    return cita


def cancelar_cita(cita):
    """Cancela una cita pendiente."""
    for c in CITAS:
        if c["id"] == cita:
            if c["estado"] != "pendiente":
                return {"error": f"La cita {cita} está «{c['estado']}»."}
            c["estado"] = "cancelada"
            return {"cancelada": cita, "fecha": c["fecha"], "hora": c["hora"]}
    return {"error": f"No existe la cita {cita}."}


def avisar(dueno, texto):
    """Envía un aviso al dueño (aquí se guarda en una lista)."""
    if dueno not in DUENOS:
        return {"error": f"No existe el cliente {dueno}."}
    AVISOS.append({"dueno": dueno, "texto": texto})
    return {"enviado": True, "a": DUENOS[dueno]["nombre"],
            "por": DUENOS[dueno]["avisos"]}

In [ ]:
print(get_pac("Lola"))
print(json.dumps(get_pac("M-012"), ensure_ascii=False)[:600], "…")
print(len(tok(json.dumps(get_pac("M-012"), ensure_ascii=False))["input_ids"]), "tokens")

## 3. Los esquemas y el agente

`automatico` genera los esquemas a partir de la firma y el comentario de cada función, como muchas bibliotecas. El agente sigue el esquema ReAct: un pensamiento y una herramienta en cada paso.

In [ ]:
def herramienta(nombre, descripcion, parametros):
    """Esquema de una herramienta; los parámetros sin valor por defecto son obligatorios."""
    return {"type": "function", "function": {
        "name": nombre, "description": descripcion,
        "parameters": {"type": "object",
                       "required": [p for p, (_, oblig) in parametros.items() if oblig],
                       "properties": {p: {"type": "string", "description": d}
                                      for p, (d, _) in parametros.items()}}}}


def automatico(funciones):
    """Esquemas generados de la firma y el docstring, como hacen muchas bibliotecas."""
    import inspect
    return [herramienta(n, (inspect.getdoc(f) or "").strip(),
                        {p: ("", True) for p in inspect.signature(f).parameters})
            for n, f in funciones.items()]


SISTEMA = (
    "You are the assistant of the staff of Clínica Veterinaria Naranco, a veterinary "
    "clinic in Oviedo (Spain). Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in Spanish, in one or two sentences.")

In [ ]:
for e in automatico(JUEGO_API)[:3]:
    print(json.dumps(e, ensure_ascii=False))

In [ ]:
def generar(mensajes, esquemas, max_tokens=400):
    """Una respuesta del modelo y los tokens de entrada y de salida."""
    texto = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False,
                                    add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    return (tok.decode(nuevos, skip_special_tokens=True).strip(),
            entrada["input_ids"].shape[1], len(nuevos))


def leer_llamadas(respuesta):
    """Las llamadas que pide el modelo, escritas entre <tool_call> y </tool_call>."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)</tool_call>", respuesta, re.S):
        try:
            llamada = json.loads(bloque)
            llamadas.append({"name": llamada["name"],
                             "arguments": llamada.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def ejecutar(llamada, funciones):
    """Ejecuta una llamada si la herramienta existe y los argumentos encajan."""
    funcion = funciones.get(llamada["name"])
    if funcion is None:
        return {"error": f"No existe la herramienta {llamada['name']}."}
    try:
        return funcion(**llamada["arguments"])
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Argumentos no válidos: {e}"}


def agente(tarea, funciones, esquemas, max_pasos=8, ver=False):
    """Pensar, actuar, observar y repetir. Cuenta los tokens de los resultados."""
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": tarea}]
    traza = []
    for paso in range(1, max_pasos + 1):
        respuesta, entrada, salida = generar(mensajes, esquemas)
        llamadas = leer_llamadas(respuesta)[:1]            # una cada vez
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        traza.append({"paso": paso, "tokens_entrada": entrada, "tokens_salida": salida,
                      "pensamiento": pensamiento, "llamadas": []})
        if not llamadas:
            traza[-1]["respuesta"] = respuesta
            if ver:
                print(f"[{paso}] Respuesta: {respuesta}")
            return respuesta, traza
        if ver and pensamiento:
            print(f"[{paso}] {pensamiento}")
        mensajes.append({"role": "assistant", "content": pensamiento, "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            resultado = ejecutar(llamada, funciones)
            texto = json.dumps(resultado, ensure_ascii=False)
            traza[-1]["llamadas"].append({**llamada, "resultado": resultado,
                                          "tokens_resultado": len(tok(texto)["input_ids"])})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": texto})
            if ver:
                print(f"[{paso}] {llamada['name']}({json.dumps(llamada['arguments'], ensure_ascii=False)})")
                print(f"     → {texto[:150]}")
    return None, traza

In [ ]:
tarea = "¿Cuándo le toca a Lola, la perra de Marta Fernández, la próxima vacuna de la rabia?"
reiniciar()
respuesta, traza = agente(tarea, JUEGO_API, automatico(JUEGO_API), ver=True)

## 4. Paso 2: nombres y descripciones

Las mismas funciones con los mismos resultados; solo cambia lo que el modelo lee de ellas.

In [ ]:
def ficha_mascota(id_mascota):
    return get_pac(id_mascota)


def ficha_cliente(id_cliente):
    return get_cli(id_cliente)


def buscar(tabla, campo, valor):
    return qry(tabla, campo, valor)


def agenda(fecha):
    return agenda_dia(fecha)


JUEGO_NOMBRES = {"ficha_mascota": ficha_mascota, "ficha_cliente": ficha_cliente,
                 "buscar": buscar, "agenda": agenda, "reservar_cita": reservar_cita,
                 "cancelar_cita": cancelar_cita, "avisar_dueno": avisar}


ESQUEMAS_NOMBRES = [
    herramienta("ficha_mascota", "Ficha de una mascota de la clínica: datos, dueño, "
                "vacunas e historial clínico.", {"id_mascota": ("Identificador, como M-001.", True)}),
    herramienta("ficha_cliente", "Ficha de un cliente (dueño): datos de contacto, sus "
                "mascotas y sus facturas.", {"id_cliente": ("Identificador, como D-001.", True)}),
    herramienta("buscar", "Busca registros cuyo campo contenga un valor. Úsala para "
                "encontrar el identificador de una mascota, un dueño o un medicamento a "
                "partir de su nombre.",
                {"tabla": ("mascotas, duenos o medicamentos.", True),
                 "campo": ("Campo en el que buscar, como nombre.", True),
                 "valor": ("Texto que buscar.", True)}),
    herramienta("agenda", "Citas de la clínica en un día.",
                {"fecha": ("Fecha en formato AAAA-MM-DD.", True)}),
    herramienta("reservar_cita", "Reserva una cita en un hueco libre.",
                {"mascota": ("Identificador de la mascota, como M-001.", True),
                 "veterinario": ("Identificador del veterinario: V1 (Dra. Ana Suárez), "
                                 "V2 (Dr. Pablo Menéndez) o V3 (Dra. Lucía Fano).", True),
                 "fecha": ("AAAA-MM-DD.", True), "hora": ("HH:MM, en punto o y media.", True),
                 "motivo": ("Motivo de la cita.", True)}),
    herramienta("cancelar_cita", "Cancela una cita pendiente.",
                {"cita": ("Identificador de la cita, como CT-0026.", True)}),
    herramienta("avisar_dueno", "Envía un aviso al dueño de una mascota (correo o SMS, "
                "según prefiera).", {"dueno": ("Identificador del cliente, como D-001.", True),
                                     "texto": ("Texto del aviso, en español.", True)}),
]

In [ ]:
reiniciar()
respuesta, traza = agente(tarea, JUEGO_NOMBRES, ESQUEMAS_NOMBRES, ver=True)

## 5. Paso 3: resultados para el modelo

Los mismos nombres y esquemas; cada herramienta devuelve solo lo que sirve para decidir, con las cuentas hechas y los errores explicados.

In [ ]:
def edad(nacimiento):
    """Años cumplidos el día de los datos."""
    n = date.fromisoformat(nacimiento)
    return HOY.year - n.year - ((HOY.month, HOY.day) < (n.month, n.day))


def resumen_mascota(mid):
    """Lo esencial de una mascota, sin el historial completo."""
    m = MASCOTAS[mid]
    return {"id": mid, "nombre": m["nombre"], "especie": m["especie"], "raza": m["raza"],
            "edad_anos": edad(m["nacimiento"]), "peso_actual_kg": m["peso_kg"],
            "alergias": m["alergias"] or "ninguna conocida",
            "dueno": {"id": m["dueno"], "nombre": DUENOS[m["dueno"]]["nombre"]}}


def vacunas_con_estado(mid):
    """Cada vacuna con su próxima fecha y si ya está vencida."""
    return [{"vacuna": v["vacuna"], "proxima": v["proxima"],
             "estado": "vencida" if v["proxima"] < HOY.isoformat() else "al día"}
            for v in VACUNAS[mid]]


def ficha_mascota_3(id_mascota):
    if id_mascota not in MASCOTAS:
        return {"error": f"No existe la mascota {id_mascota}. Búscala por nombre con "
                         "buscar(tabla='mascotas', campo='nombre', valor=...)."}
    ultima = HISTORIAL[id_mascota][-1] if HISTORIAL[id_mascota] else None
    return {**resumen_mascota(id_mascota), "vacunas": vacunas_con_estado(id_mascota),
            "ultima_visita": ultima and {k: ultima[k] for k in
                                         ("fecha", "motivo", "tratamiento", "peso_kg")},
            "visitas_en_historial": len(HISTORIAL[id_mascota])}


def ficha_cliente_3(id_cliente):
    if id_cliente not in DUENOS:
        return {"error": f"No existe el cliente {id_cliente}. Búscalo por nombre con "
                         "buscar(tabla='duenos', campo='nombre', valor=...)."}
    pendientes = [f for f in FACTURAS if f["dueno"] == id_cliente and not f["pagada"]]
    return {"id": id_cliente, "nombre": DUENOS[id_cliente]["nombre"],
            "avisos_por": DUENOS[id_cliente]["avisos"],
            "mascotas": [{"id": m, "nombre": d["nombre"], "especie": d["especie"]}
                         for m, d in MASCOTAS.items() if d["dueno"] == id_cliente],
            "facturas_sin_pagar": [{"id": f["id"], "fecha": f["fecha"], "total": f["total"]}
                                   for f in pendientes],
            "total_sin_pagar": round(sum(f["total"] for f in pendientes), 2)}


def buscar_3(tabla, campo, valor):
    r = qry(tabla, campo, valor)
    if r["status"] != 200:
        return {"error": f"No existe la tabla «{tabla}». Las tablas son mascotas, "
                         "duenos y medicamentos."}
    if not r["rows"]:
        return {"error": f"Ningún registro de {tabla} tiene «{valor}» en {campo}.",
                "resultados": []}
    if tabla == "mascotas":
        return {"resultados": [resumen_mascota(f["id"]) for f in r["rows"]]}
    if tabla == "duenos":
        return {"resultados": [{"id": f["id"], "nombre": f["nombre"]} for f in r["rows"]]}
    return {"resultados": [{"nombre": f["id"], "presentacion": f["presentacion"],
                            "dosis_mg_por_kg": f["dosis_mg_kg"], "existencias": f["existencias"],
                            "reposicion": f.get("reposicion", "sin fecha")}
                           for f in r["rows"]]}


def agenda_3(fecha):
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(fecha)):
        return {"error": f"La fecha «{fecha}» no es válida; usa AAAA-MM-DD."}
    citas = [c for c in CITAS if c["fecha"] == fecha and c["estado"] != "cancelada"]
    return {"fecha": fecha, "citas": [f"{c['id']} {c['hora']} {c['veterinario']} "
                                      f"{MASCOTAS[c['mascota']]['nombre']} ({c['mascota']})"
                                      for c in citas],
            "veterinarios": VETERINARIOS,
            "horas_libres": {v: [h for h in HORAS if not any(
                c["hora"] == h and c["veterinario"] == v for c in citas)]
                for v in VETERINARIOS}}


JUEGO_RESULTADOS = {"ficha_mascota": ficha_mascota_3, "ficha_cliente": ficha_cliente_3,
                    "buscar": buscar_3, "agenda": agenda_3, "reservar_cita": reservar_cita,
                    "cancelar_cita": cancelar_cita, "avisar_dueno": avisar}


ESQUEMAS_RESULTADOS = ESQUEMAS_NOMBRES          # mismos esquemas, otros resultados

In [ ]:
n = lambda r: len(tok(json.dumps(r, ensure_ascii=False))["input_ids"])
print("Ficha de Zeus:", n(get_pac("M-012")), "tokens →", n(ficha_mascota_3("M-012")))
print("Agenda del 21:", n(agenda_dia("2026-10-21")), "tokens →", n(agenda_3("2026-10-21")))
print(ficha_mascota_3("M-002")["vacunas"])

## 6. Paso 4: herramientas con la forma de las tareas

Once herramientas que hacen un trabajo completo y reciben los nombres que usa el personal, no identificadores internos.

In [ ]:
def localizar(mascota, dueno=""):
    """El id de la mascota a partir de su nombre (y el del dueño, si hay varias)."""
    ids = [m for m, d in MASCOTAS.items() if sin_tildes(mascota) == sin_tildes(d["nombre"])
           and sin_tildes(dueno) in sin_tildes(DUENOS[d["dueno"]]["nombre"])]
    if not ids:
        return None, {"error": f"No hay ninguna mascota llamada «{mascota}»"
                               + (f" de {dueno}" if dueno else "") + " en la clínica."}
    if len(ids) > 1:
        return None, {"error": f"Hay {len(ids)} mascotas llamadas «{mascota}»; indica el "
                               "nombre del dueño.",
                      "mascotas": [resumen_mascota(m) for m in ids]}
    return ids[0], None


def veterinario_de(nombre):
    """El id del veterinario a partir de su nombre o de su id."""
    for v, n in VETERINARIOS.items():
        if sin_tildes(nombre) in (sin_tildes(v), sin_tildes(n)) or \
                sin_tildes(nombre) in sin_tildes(n):
            return v
    return None


def buscar_mascota(nombre, dueno=""):
    mid, error = localizar(nombre, dueno)
    return error or resumen_mascota(mid)


def mascotas_de(dueno):
    ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
    if not ids:
        return {"error": f"No hay ningún cliente llamado «{dueno}»."}
    return {"clientes": [{"nombre": DUENOS[d]["nombre"], "mascotas": [
        {"nombre": x["nombre"], "especie": x["especie"], "raza": x["raza"]}
        for m, x in MASCOTAS.items() if x["dueno"] == d]} for d in ids]}


def vacunas(mascota, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(),
                     "vacunas": vacunas_con_estado(mid)}


def historial(mascota, ultimas=3, dueno=""):
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    return {"mascota": MASCOTAS[mid]["nombre"], "visitas": [
        {k: v[k] for k in ("fecha", "motivo", "tratamiento", "peso_kg")}
        for v in HISTORIAL[mid][-int(ultimas):]]}


def dosis(medicamento, mascota, dueno=""):
    med = FARMACIA.get(sin_tildes(medicamento))
    if med is None:
        return {"error": f"No tenemos «{medicamento}». Medicamentos: "
                         + ", ".join(FARMACIA) + "."}
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    m = MASCOTAS[mid]
    avisos = []
    if sin_tildes(medicamento) in [sin_tildes(a) for a in m["alergias"]]:
        avisos.append(f"{m['nombre']} es alérgico a {medicamento}: no recetar.")
    if m["especie"] not in med["especies"]:
        avisos.append(f"{medicamento} no está indicado en {m['especie']}s.")
    return {"medicamento": sin_tildes(medicamento), "mascota": m["nombre"],
            "peso_kg": m["peso_kg"], "dosis_mg_por_kg": med["dosis_mg_kg"],
            "dosis_mg_por_toma": round(med["dosis_mg_kg"] * m["peso_kg"], 2),
            "existencias": med["existencias"],
            "reposicion": med.get("reposicion", "sin fecha") if not med["existencias"] else None,
            "avisos": avisos or "ninguno"}


def huecos_libres(fecha, veterinario=""):
    r = agenda_3(fecha)
    if "error" in r:
        return r
    vets = [veterinario_de(veterinario)] if veterinario else list(VETERINARIOS)
    if vets == [None]:
        return {"error": f"No hay ningún veterinario «{veterinario}». Son: "
                         + ", ".join(VETERINARIOS.values()) + "."}
    return {"fecha": fecha, "horas_libres": {VETERINARIOS[v]: r["horas_libres"][v]
                                             for v in vets}}


def citas_de(mascota, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "citas": [
        {"id": c["id"], "fecha": c["fecha"], "hora": c["hora"], "motivo": c["motivo"],
         "veterinario": VETERINARIOS[c["veterinario"]], "estado": c["estado"]}
        for c in CITAS if c["mascota"] == mid]}


def cuentas(dueno):
    ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
    if not ids:
        return {"error": f"No hay ningún cliente llamado «{dueno}»."}
    return {"clientes": [{k: v for k, v in ficha_cliente_3(d).items() if k != "id"}
                         for d in ids]}


def reservar(mascota, veterinario, fecha, hora, motivo, dueno=""):
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    vet = veterinario_de(veterinario)
    if vet is None:
        return {"error": f"No hay ningún veterinario «{veterinario}». Son: "
                         + ", ".join(VETERINARIOS.values()) + "."}
    r = reservar_cita(mid, vet, fecha, hora, motivo)
    if "error" in r:
        return r
    return {"reservada": r["id"], "mascota": MASCOTAS[mid]["nombre"],
            "veterinario": VETERINARIOS[vet], "fecha": fecha, "hora": hora}


def avisar_dueno_de(mascota, texto, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or avisar(MASCOTAS[mid]["dueno"], texto)


JUEGO_TAREAS = {"buscar_mascota": buscar_mascota, "mascotas_de": mascotas_de,
                "vacunas": vacunas, "historial": historial, "dosis": dosis,
                "huecos_libres": huecos_libres, "citas_de": citas_de, "cuentas": cuentas,
                "reservar_cita": reservar, "cancelar_cita": cancelar_cita,
                "avisar_dueno": avisar_dueno_de}

In [ ]:
MASCOTA = ("Nombre de la mascota.", True)


DUENO = ("Nombre del dueño; solo si hay varias mascotas con ese nombre.", False)


ESQUEMAS_TAREAS = [
    herramienta("buscar_mascota", "Datos de una mascota buscada por su nombre: especie, "
                "edad, peso actual, alergias y dueño.", {"nombre": MASCOTA, "dueno": DUENO}),
    herramienta("mascotas_de", "Las mascotas de un cliente, buscado por su nombre.",
                {"dueno": ("Nombre del dueño.", True)}),
    herramienta("vacunas", "Vacunas de una mascota: la próxima fecha de cada una y si "
                "está vencida.", {"mascota": MASCOTA, "dueno": DUENO}),
    herramienta("historial", "Las últimas visitas de una mascota: fecha, motivo, "
                "tratamiento y peso.", {"mascota": MASCOTA,
                                        "ultimas": ("Cuántas visitas (por defecto, 3).", False),
                                        "dueno": DUENO}),
    herramienta("dosis", "Dosis de un medicamento para una mascota según su peso actual, "
                "con avisos de alergia o especie y las existencias.",
                {"medicamento": ("Nombre del medicamento.", True), "mascota": MASCOTA,
                 "dueno": DUENO}),
    herramienta("huecos_libres", "Horas libres de la agenda en un día, de un veterinario o "
                "de todos.", {"fecha": ("AAAA-MM-DD.", True),
                              "veterinario": ("Nombre del veterinario; vacío para todos.", False)}),
    herramienta("citas_de", "Citas de una mascota (pasadas y pendientes), con su id.",
                {"mascota": MASCOTA, "dueno": DUENO}),
    herramienta("cuentas", "Facturas sin pagar de un cliente, buscado por su nombre, y el "
                "total que debe.", {"dueno": ("Nombre del dueño.", True)}),
    herramienta("reservar_cita", "Reserva una cita en una hora libre.",
                {"mascota": MASCOTA, "veterinario": ("Nombre del veterinario.", True),
                 "fecha": ("AAAA-MM-DD.", True), "hora": ("HH:MM, en punto o y media.", True),
                 "motivo": ("Motivo de la cita.", True), "dueno": DUENO}),
    herramienta("cancelar_cita", "Cancela una cita pendiente.",
                {"cita": ("Id de la cita, como CT-0026 (lo da citas_de).", True)}),
    herramienta("avisar_dueno", "Envía un aviso al dueño de una mascota, por correo o SMS "
                "según prefiera.", {"mascota": MASCOTA, "texto": ("Texto del aviso, en "
                                                                   "español.", True),
                                     "dueno": DUENO}),
]

In [ ]:
print(dosis("meloxicam", "Max"))
print(dosis("amoxicilina", "Simba"))
print(huecos_libres("2026-10-21", "Ana Suárez"))
print(buscar_mascota("Pancho"))

## 7. Doce tareas y cuatro juegos de herramientas

Cada tarea se comprueba por la respuesta (fechas, importes, nombres) o por el estado de la clínica: las citas creadas o canceladas y los avisos enviados.

In [ ]:
def dice(respuesta, *formas):
    """¿Aparece en la respuesta alguna de esas formas?"""
    return any(re.search(f, respuesta or "", re.I) for f in formas)


def fecha_en(respuesta, iso):
    """¿Aparece la fecha, escrita como 2027-08-22, 22/08/2027 o 22 de agosto de 2027?"""
    a, m, d = iso.split("-")
    meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
             "septiembre", "octubre", "noviembre", "diciembre"]
    return dice(respuesta, iso, rf"\b0?{int(d)}/0?{int(m)}/{a}",
                rf"\b0?{int(d)} de {meses[int(m) - 1]}( de {a})?")


def citas_nuevas():
    return [c for c in CITAS if c["id"] not in {x["id"] for x in CITAS0}]


def primera_libre(fecha, vet):
    ocupadas = {c["hora"] for c in CITAS0 if c["fecha"] == fecha and c["veterinario"] == vet}
    return next(h for h in HORAS if h not in ocupadas)


def debe(nombre):
    return round(sum(f["total"] for f in FACTURAS if not f["pagada"]
                     and DUENOS[f["dueno"]]["nombre"] == nombre), 2)


def importe(respuesta, valor):
    """¿Aparece el importe, como 136, 136,00 o 136.00?"""
    entero = f"{valor:.2f}".replace(".00", "")
    return dice(respuesta, rf"\b{entero}([,.]00?)?\b", f"{valor:.2f}".replace(".", ","))


def aviso_a(nombre, *formas):
    return any(DUENOS[a["dueno"]]["nombre"] == nombre
               and all(re.search(f, a["texto"], re.I) for f in formas) for a in AVISOS)


TAREAS = [
    ("vacuna_rabia", "¿Cuándo le toca a Lola, la perra de Marta Fernández, la próxima "
     "vacuna de la rabia?",
     lambda r: fecha_en(r, "2027-08-22")),
    ("dosis", "¿Cuántos miligramos de meloxicam le corresponden a Max en cada toma?",
     lambda r: dice(r, r"2[,.]07", r"2[,.]1\b")),
    ("alergia", "Van a recetarle amoxicilina a Simba. ¿Hay algún problema?",
     lambda r: dice(r, r"al[eé]rg")),
    ("reservar", "Reserva para Toby una revisión con el Dr. Pablo Menéndez el jueves 22, "
     "en la primera hora libre.",
     lambda r: [(c["mascota"], c["veterinario"], c["fecha"], c["hora"]) for c in citas_nuevas()]
     == [("M-002", "V2", "2026-10-22", primera_libre("2026-10-22", "V2"))]),
    ("cancelar", "Cancela la cita que tiene Rocky mañana.",
     lambda r: [c["id"] for c in CITAS if c["estado"] == "cancelada"] == ["CT-0060"]),
    ("vencidas_aviso", "Avisa al dueño de Toby de las vacunas que tiene vencidas.",
     lambda r: aviso_a("Pedro Álvarez", "rabia", "polivalente", "leishmaniosis")),
    ("deuda", "¿Cuánto debe Óscar Prieto en facturas sin pagar?",
     lambda r: importe(r, debe("Óscar Prieto"))),
    ("gata_de_marta", "Marta Fernández llama por su gata. ¿Tiene alguna vacuna vencida?",
     lambda r: dice(r, "trivalente") and dice(r, "vencid|caducad|atrasad|pasad")),
    ("peso", "¿Cuánto pesaba Zeus en su última visita, y cuánto ha cambiado respecto a "
     "la anterior?",
     lambda r: dice(r, r"32[,.]2") and dice(r, r"0[,.]6")
     and not dice(r, r"últim[ao] visita[^.,]{0,30}(2025|31[,.]6)", r"31[,.]6 kg en (su|la) últim")),
    ("ultima_visita", "¿Por qué vino Nube la última vez y qué tratamiento le pusieron?",
     lambda r: dice(r, "herida") and dice(r, "amoxicilina")),
    ("huecos", "¿Qué horas tiene libres la Dra. Ana Suárez el miércoles 21 por la tarde?",
     lambda r: all(dice(r, h) for h in ("16:00", "16:30", "17:30", "18:00", "18:30"))
     and not dice(r, r"17:00")),
    ("no_existe", "¿Cuándo le toca la vacuna de la rabia a Pancho?",
     lambda r: dice(r, r"no (hay|existe|encuentr|figura|tenemos|aparece)", "ning[uú]n")
     and not dice(r, r"\b20\d\d-\d\d-\d\d\b")),
]

In [ ]:
def resolver(funciones, esquemas, tareas=TAREAS, max_pasos=8):
    """Resuelve cada tarea con la clínica reiniciada y anota el resultado."""
    filas = []
    for clave, tarea, comprobar in tareas:
        reiniciar()
        inicio = time.time()
        respuesta, traza = agente(tarea, funciones, esquemas, max_pasos)
        llamadas = [l for p in traza for l in p["llamadas"]]
        filas.append({
            "tarea": clave, "bien": bool(comprobar(respuesta)), "respuesta": respuesta,
            "pasos": len(traza), "llamadas": [(l["name"], l["arguments"]) for l in llamadas],
            "errores": sum("error" in l["resultado"] or l["resultado"].get("status", 200) != 200
                           for l in llamadas if isinstance(l["resultado"], dict)),
            "tokens_resultados": sum(l["tokens_resultado"] for l in llamadas),
            "tokens_entrada": sum(p["tokens_entrada"] for p in traza),
            "segundos": round(time.time() - inicio, 1), "traza": traza,
            "citas_nuevas": citas_nuevas(), "avisos": AVISOS})
    return filas


def resumen(filas):
    return {"bien": sum(f["bien"] for f in filas),
            "terminadas": sum(f["respuesta"] is not None for f in filas),
            "pasos": sum(f["pasos"] for f in filas),
            "llamadas": sum(len(f["llamadas"]) for f in filas),
            "errores": sum(f["errores"] for f in filas),
            "tokens_resultados": sum(f["tokens_resultados"] for f in filas),
            "tokens_entrada": sum(f["tokens_entrada"] for f in filas),
            "segundos": round(sum(f["segundos"] for f in filas), 1)}


def juegos():
    """Los cuatro juegos de herramientas del capítulo, en el orden en que se construyen."""
    return {"La API tal cual": (JUEGO_API, automatico(JUEGO_API)),
            "Nombres y descripciones": (JUEGO_NOMBRES, ESQUEMAS_NOMBRES),
            "Resultados para el modelo": (JUEGO_RESULTADOS, ESQUEMAS_RESULTADOS),
            "Herramientas con forma de tarea": (JUEGO_TAREAS, ESQUEMAS_TAREAS)}

In [ ]:
resultados = {}
for nombre, (funciones, esquemas) in juegos().items():
    resultados[nombre] = resolver(funciones, esquemas)
    print(f"{nombre:32s}", resumen(resultados[nombre]))

Las tareas de cada juego, con una marca en las bien hechas:

In [ ]:
for nombre, filas in resultados.items():
    print(f"{nombre:32s}", " ".join(("✓" if f["bien"] else "✗") + f["tarea"] for f in filas))

La tarea del peso de Zeus en los pasos 2, 3 y 4: qué llama el modelo, cuánto ocupa cada resultado y qué responde.

In [ ]:
for nombre in list(resultados)[1:]:
    f = next(f for f in resultados[nombre] if f["tarea"] == "peso")
    print("---", nombre, "✓" if f["bien"] else "✗")
    for paso in f["traza"]:
        for l in paso["llamadas"]:
            print("   ", l["name"], l["arguments"], "→", l["tokens_resultado"], "tokens")
    print("    Respuesta:", f["respuesta"])

## Ejercicios

1. Explica con tus palabras las cuatro partes de la interfaz de una herramienta. ¿Cuál de ellas cambia en cada paso del capítulo?

2. Imprime el esquema que genera `automatico` para `qry` y el que se escribe a mano para `buscar`. ¿Cuántos tokens ocupa cada uno? ¿Qué información tiene el segundo que el modelo no podía deducir del primero?

3. Llama a mano, sin el modelo, a `get_pac` y a `ficha_mascota_3` con la ficha de Zeus, `M-012`, y a `historial` con su nombre. Cuenta los tokens de cada resultado con el tokenizador. ¿Qué campos quitarías aún de la ficha del juego 3?

4. Cambia solo la descripción de `qry` en el juego 1 para que diga qué tablas existen y la forma de los identificadores. ¿Cuántas tareas recupera? ¿Qué dice eso del paso 2?

5. En el juego 3, la tarea del peso de Zeus falla porque la ficha trae solo la última visita. Haz que traiga las dos últimas. ¿La resuelve? ¿Cambia alguna otra?

6. Añade a `agenda_3` un parámetro opcional `veterinario`, como el de `huecos_libres`. ¿Recupera el juego 3 la tarea de las horas libres?

7. Escribe tres tareas nuevas para la clínica, con su comprobación: una que necesite dos herramientas del juego 4, otra para una clienta con dos mascotas y otra que pida una acción imposible. ¿Cuántas resuelve el agente?

8. La tarea de cancelar falla en el juego 4 porque el modelo confunde hoy con mañana. Prueba dos arreglos: que `citas_de` marque cada cita con «hoy», «mañana» o «pasada», o que `cancelar_cita` reciba la mascota y la fecha en lugar del identificador. ¿Cuál prefieres, y por qué?

9. Une `vacunas`, `historial` y `citas_de` en una sola herramienta `ficha_completa(mascota)`. Mide el acierto y los tokens. ¿Dónde está el punto medio entre pocas herramientas grandes y muchas pequeñas?

10. **Proyecto.** Elige una API que conozcas (la de una biblioteca, un servicio de tu trabajo o una que inventes) y diseña tres o cuatro herramientas sobre ella para un modelo. Escribe seis tareas con su comprobación, mide la API tal cual y tus herramientas, y redacta un breve informe con lo que cambiaste y lo que mejoró.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Writing effective tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents) (Anthropic, 2025). Lección 13.01 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La Clínica Veterinaria Naranco y sus datos son ficticios. Código y experimentos propios.